In [ ]:

import pandas as pd
import numpy as np
import os

PROCESSED_DIR = "../data/processed/"

train_df = pd.read_csv(PROCESSED_DIR + "train_clean.csv")
test_df  = pd.read_csv(PROCESSED_DIR + "test_clean.csv")


y_train = train_df["Survived"].copy()
X_train = train_df.drop(columns=["Survived"])
X_test  = test_df.copy()

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)

print("\nColumn dtypes:")
print(X_train.dtypes)

print("\nAny missing?")
print("Train:", X_train.isnull().sum().sum())
print("Test :", X_test.isnull().sum().sum())

X_train shape: (891, 11)
X_test shape : (418, 11)
y_train shape: (891,)

Column dtypes:
Pclass          int64
Sex               str
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked          str
Title             str
FamilySize      int64
IsAlone         int64
HasCabin        int64
dtype: object

Any missing?
Train: 0
Test : 1


In [4]:
# Check what's missing in test
print("=== Test Missing Values ===")
print(X_test.isnull().sum()[X_test.isnull().sum() > 0])

=== Test Missing Values ===
Fare    1
dtype: int64


In [6]:

fare_median = X_train["Fare"].median()
X_test["Fare"] = X_test["Fare"].fillna(fare_median)

print("Fare median used for imputation:", round(fare_median, 2))
print("Test missing after fix:", X_test.isnull().sum().sum())

Fare median used for imputation: 14.45
Test missing after fix: 0


In [8]:
print("Train missing:", X_train.isnull().sum())
print("Test missing :", X_test.isnull().sum())

Train missing: Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
Title         0
FamilySize    0
IsAlone       0
HasCabin      0
dtype: int64
Test missing : Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
Title         0
FamilySize    0
IsAlone       0
HasCabin      0
dtype: int64


In [9]:

from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("X_tr shape :", X_tr.shape)
print("X_val shape:", X_val.shape)
print("y_tr shape :", y_tr.shape)
print("y_val shape:", y_val.shape)

print("\nClass balance — Train:")
print(y_tr.value_counts(normalize=True).round(4))
print("\nClass balance — Validation:")
print(y_val.value_counts(normalize=True).round(4))

X_tr shape : (712, 11)
X_val shape: (179, 11)
y_tr shape : (712,)
y_val shape: (179,)

Class balance — Train:
Survived
0    0.6166
1    0.3834
Name: proportion, dtype: float64

Class balance — Validation:
Survived
0    0.6145
1    0.3855
Name: proportion, dtype: float64


In [11]:

numeric_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone", "HasCabin"
]

# Categorical columns — must be encoded into numbers
categorical_features = [
    "Sex", "Embarked", "Title"
]

print("Numeric features   :", numeric_features)
print("Categorical features:", categorical_features)

print("\nSanity check:")
print(f"  Numeric count     : {len(numeric_features)}")
print(f"  Categorical count : {len(categorical_features)}")
print(f"  Total             : {len(numeric_features) + len(categorical_features)}")
print(f"  Actual columns    : {X_train.shape[1]}")

Numeric features   : ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'HasCabin']
Categorical features: ['Sex', 'Embarked', 'Title']

Sanity check:
  Numeric count     : 8
  Categorical count : 3
  Total             : 11
  Actual columns    : 11


In [13]:
# Block 4: Build the ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Numeric pipeline: impute (safety) → scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline: impute (safety) → one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines with the ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessor built:")
print(preprocessor)

Preprocessor built:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare',
                                  'FamilySize', 'IsAlone', 'HasCabin']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Sex', 'Embarked', 'Title'])])


In [15]:

from sklearn.linear_model import LogisticRegression

full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

print("Full pipeline:")
print(full_pipeline)

Full pipeline:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare',
                                                   'FamilySize', 'IsAlone',
                                                   'HasCabin']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                    

In [17]:
X_tr_processed  = preprocessor.fit_transform(X_tr)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("=== Shapes After Preprocessing ===")
print(f"X_tr_processed   : {X_tr_processed.shape}")
print(f"X_val_processed  : {X_val_processed.shape}")
print(f"X_test_processed : {X_test_processed.shape}")

print(f"\nOriginal X_tr columns : {X_tr.shape[1]}")
print(f"Processed columns      : {X_tr_processed.shape[1]}")

=== Shapes After Preprocessing ===
X_tr_processed   : (712, 18)
X_val_processed  : (179, 18)
X_test_processed : (418, 18)

Original X_tr columns : 11
Processed columns      : 18


In [18]:
print("=== 1. Shape Check ===")
print(f"X_tr_processed   : {X_tr_processed.shape}")
print(f"X_val_processed  : {X_val_processed.shape}")
print(f"X_test_processed : {X_test_processed.shape}")

# Same number of columns everywhere?
assert X_tr_processed.shape[1] == X_val_processed.shape[1] == X_test_processed.shape[1], \
    "Column mismatch between train/val/test!"
print("All three have the same number of columns.")

print("\n=== 2. NaN Check ===")
print(f"NaNs in train : {np.isnan(X_tr_processed).sum()}")
print(f"NaNs in val   : {np.isnan(X_val_processed).sum()}")
print(f"NaNs in test  : {np.isnan(X_test_processed).sum()}")

print("\n=== 3. Type Check ===")
print(f"Data type: {X_tr_processed.dtype}")
print(" All numeric." if X_tr_processed.dtype != object else " Contains non-numeric data!")

print("\n=== 4. Sneak Peek — Fit Full Pipeline ===")
full_pipeline.fit(X_tr, y_tr)
y_val_pred = full_pipeline.predict(X_val)

from sklearn.metrics import accuracy_score
val_acc = accuracy_score(y_val, y_val_pred)
print(f"Validation accuracy (Logistic Regression): {val_acc:.4f}")
print(f"Baseline (always predict 'died'): {(y_val == 0).mean():.4f}")

=== 1. Shape Check ===
X_tr_processed   : (712, 18)
X_val_processed  : (179, 18)
X_test_processed : (418, 18)
All three have the same number of columns.

=== 2. NaN Check ===
NaNs in train : 0
NaNs in val   : 0
NaNs in test  : 0

=== 3. Type Check ===
Data type: float64
 All numeric.

=== 4. Sneak Peek — Fit Full Pipeline ===
Validation accuracy (Logistic Regression): 0.8436
Baseline (always predict 'died'): 0.6145
